In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas import read_csv

Download tables of results from **Kraken**  `species_abundance`  
Make a taxon an index  
Join tables by taxon

In [ ]:
df_il = read_csv('../data/kraken_il_species_abundance.tsv', sep='\t')
df_he = read_csv('../data/kraken_he_species_abundance.tsv', sep='\t')
df_he = df_he.set_index("taxon")
df_il = df_il.set_index('taxon')
df_all = pd.concat([df_il, df_he], axis=1).fillna(0)

Create metadata with sample group 

In [ ]:
meta_il = pd.DataFrame({'sample': df_il.columns, 'group': 'disease'})
meta_he = pd.DataFrame({'sample': df_he.columns, 'group': 'healthy'})
meta = pd.concat([meta_il, meta_he], ignore_index=True)
meta = meta.set_index('sample')

Converting the number of reads to a relative abundance in the sample

In [ ]:
df_rel = df_all.div(df_all.sum(axis=0), axis=1) 

Create top taxa  
Group by healthy/disease.  
For average abundance by group

In [ ]:
top_taxa = df_rel.sum(axis=1).sort_values(ascending=False).head(10).index
df_top = df_rel.loc[top_taxa]
other = 1 - df_top.sum(axis=0)
df_top.loc["Other"] = other
df_top_t = df_top.T
df_top_t["group"] = meta["group"]
grouped_top = df_top_t.groupby("group").mean()
grouped_top = grouped_top.drop(columns="group", errors="ignore")

Plot for Mean relative abundance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

grouped_top.plot(kind="bar", stacked=True, ax=ax, width=0.3)

ax.set_ylabel("Mean relative abundance", fontsize=14)
ax.set_xlabel("Group", fontsize=14)

ax.tick_params(axis='x', labelsize=12, rotation=0)
ax.tick_params(axis='y', labelsize=12)

handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[::-1],
    labels[::-1],
    bbox_to_anchor=(0.5, 0.7),
    loc="center",
    fontsize=11,
    title="Taxa",
    title_fontsize=12
)

plt.tight_layout()
plt.savefig('../results/mean_rel_abund10_kraken.png', dpi=300)
plt.show()

PCA

In [ ]:
df_rel_t = df_rel.T

In [ ]:
from modules.pca import plot_pca

pca_df, pca = plot_pca(df_rel_t, meta, method="Kraken")